# 1. Model Selection Strategy

**Objective**: Select the best algorithm and training strategy for the Kickstarter Classifiction task.

**Key Issues to Address**:
1.  **Class Imbalance**: Are successful projects rarer than failed ones?
2.  **Algorithm Suitability**: Why Gradient Boosting over others?
3.  **Hyperparameter Tuning**: How to optimize?

## 1.1 Class Imbalance Analysis
We inspect the target distribution in the processed training set.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from app.src import config

SAVE_DIR = config.RESULTS_DIR
os.makedirs(SAVE_DIR, exist_ok=True)

# Load Processed Train Data
train_df = pd.read_csv(config.TRAIN_DATA_PATH)

plt.figure(figsize=(8, 5))
sns.countplot(x=config.TARGET_COL, data=train_df, palette='viridis')
plt.title('Class Distribution (0=Failed, 1=Success)')
plt.xlabel('Target')
plt.ylabel('Count')
plt.savefig(os.path.join(SAVE_DIR, 'class_imbalance.png'))
plt.show()

ratio = train_df[config.TARGET_COL].value_counts(normalize=True)
print(f"Class Ratio:\n{ratio}")

**Strategy**:
- If imbalance is severe (<10% positive), we would use SMOTE or `scale_pos_weight`.
- In this dataset (likely ~40-60 split or ~30-70), we use **Probability Calibration (Isotonic)** to ensure the model's output probabilities are realistic, rather than just hard re-sampling.

## 1.2 Model Justification: Why Ensemble (LGBM + XGB)?

We selected a **Voting Classifier** combining LightGBM and XGBoost because:

1.  **Tabular Data SOTA**: Gradient Boosting Trees are currently state-of-the-art for structured/tabular data.
2.  **Handling Missing Data**: Tree-based models can handle missing values natively (though we imputed them for stability).
3.  **Non-Linearity**: They capture complex interactions between features (e.g., Duration vs. Goal) better than Logistic Regression.
4.  **Ensemble Robustness**: Combining two different boosting implementations reduces variance and overfitting.

## 1.3 Hyperparameter Tuning (Optuna)

Instead of Grid Search (slow), we use **Optuna** (Bayesian Optimization) to find optimal parameters for LightGBM.
The optimization logic is encapsulated in `src/train_model.py`. It maximizes the **F1-Score** or **ROC-AUC** on Cross-Validation folds.

# 2. Execution (Training Pipeline)

We now execute the training script, which:
1.  Loads Optim parameters (or defaults).
2.  Trains the VotingClassifier.
3.  Calibrates probabilities (Isotonic).
4.  Saves the model to `models/final_model.joblib`.

In [ ]:
import sys
sys.path.append(os.path.abspath('..'))
from app.src import train_model

# Run Training
train_model.main()